# Idea2Prior: from a research idea to a Top 5

This Notebook uses the same `paperarena.run_retrieval` API as the web interface and CLI. It defaults to an offline configuration check without search or model requests.

Install from the project directory:
```bash
python -m pip install -e ".[notebook]"
```

Supply model IDs and API keys you are authorized to use. Use environment variables or `getpass` for keys; do not hard-code real credentials in code cells. English is the primary guide; optional [Chinese documentation](../README.zh-CN.md) is available.


In [ ]:
from pathlib import Path
import sys, json, time
from dataclasses import replace

ROOT = Path.cwd()
if not (ROOT / "paperarena").is_dir() and (ROOT.parent / "paperarena").is_dir():
    ROOT = ROOT.parent
if (ROOT / "paperarena").is_dir():
    sys.path.insert(0, str(ROOT))
from idea2prior import APIProfile, RunConfig, run_retrieval
from paperarena.__main__ import dataset_query
from paperarena.diagnostics import safe_error


def show_state(state):
    """Display public status and safe errors; never print raw provider exceptions."""
    summary = {key: state.get(key) for key in
               ("status", "phase", "duration_seconds", "output_dir", "run_dir")}
    error = state.get("error")
    if isinstance(error, dict):
        summary["error"] = {key: error[key] for key in
                            ("type", "code", "message", "http_status", "phase") if key in error}
    diagnostics = state.get("diagnostics")
    if isinstance(diagnostics, list) and diagnostics:
        summary["diagnostics"] = [
            {key: diagnostic[key] for key in ("type", "code", "message", "http_status", "phase") if key in diagnostic}
            for diagnostic in diagnostics[:10] if isinstance(diagnostic, dict)]
        summary["issue_count"] = state.get("issue_count", len(diagnostics))
    usage = state.get("usage")
    recent = usage.get("recent_request_errors", []) if isinstance(usage, dict) else []
    if recent:
        summary["recent_request_errors"] = [
            {key: error[key] for key in ("type", "code", "message", "http_status", "phase") if key in error}
            for error in recent[-5:] if isinstance(error, dict)]
    print(json.dumps(summary, ensure_ascii=False, indent=2))


def exception_state(exc, phase, started=None):
    actual_phase = getattr(exc, "_paperarena_phase", phase)
    error = safe_error(exc, phase=actual_phase)
    return {"status": "FAILED", "phase": error.get("phase", phase),
            "duration_seconds": round(time.monotonic() - started, 3) if started else 0.0,
            "output_dir": getattr(exc, "_paperarena_output_dir", None),
            "run_dir": None, "top5": [], "usage": None, "error": error}


## 1. Configure APIs and route models

Load the recommended native Gemini preset. Retrieval/scoring (`heavy`) and lightweight tasks (`light`) use `gemini-3.1-flash-lite`; final ranking (`expert`) uses `gemini-3.8-flash`. The previous preview retrieval model is replaced by its official stable version; original IDs are preserved in configs/gemini_previous_workflow.example.json. Confirm account access or substitute available models before real retrieval.

Profiles can independently use OpenAI, native Gemini, or an OpenAI-compatible custom endpoint. If using one model for all tasks, route all three roles to the same profile. Other examples are available in `configs/`.


In [ ]:
config = None
configuration_error = None
model_selection_error = None
try:
    config_data = json.loads((ROOT / "configs/gemini_recommended.example.json").read_text(encoding="utf-8"))
    # Optional: route every role to one OpenAI profile:
    # config_data["profiles"] = {"default": {"provider": "openai",
    #     "model": "YOUR_MODEL_ID", "api_key_env": "OPENAI_API_KEY"}}
    # config_data["roles"] = {role: "default" for role in ("heavy", "light", "expert")}
    config = RunConfig.from_dict(config_data)
except Exception as exc:
    configuration_error = exception_state(exc, "configuration")
    show_state(configuration_error)

if config is not None:
    print(json.dumps(config.to_public_dict(), ensure_ascii=False, indent=2))


### Optional: fetch available models from the selected API

`RUN_MODEL_CATALOG = False` keeps this step offline by default. Enable it explicitly to call the chosen profiles' model-list endpoints, without sending generation requests. A missing key fails before network access; unsupported catalogs, authentication failures, unavailable endpoints, and timeouts show safe diagnostic fields. Native Gemini uses its official model-list endpoint; custom profiles use the configured OpenAI-compatible API URL.

Catalog options are stored in `model_options` by profile name. The display shows the first 50 entries, while the stored list remains complete within the catalog limits. If a catalog is unavailable, you can retain or manually enter a model ID; generation availability is verified only by an actual request.


In [ ]:
RUN_MODEL_CATALOG = False
model_options = {}
catalog_errors = {}
if RUN_MODEL_CATALOG and config is not None:
    from paperarena.model_catalog import fetch_models, ModelCatalogError
    for name in sorted(set(config.roles.values())):
        started = time.monotonic()
        try:
            model_options[name] = fetch_models(config.profiles[name], timeout_seconds=15, max_pages=5)
            print(name, ":", len(model_options[name]), "model options")
            print(json.dumps(model_options[name][:50], ensure_ascii=False, indent=2))
            if not model_options[name]:
                print("No compatible model options were returned; verify the account and API protocol.")
        except ModelCatalogError as exc:
            # ModelCatalogError exposes a deliberately safe public message.
            error = {"type": type(exc).__name__, "code": exc.code,
                     "message": str(exc), "phase": "model_catalog"}
            if getattr(exc, "http_status", None) is not None:
                error["http_status"] = exc.http_status
            catalog_errors[name] = {"status": "FAILED", "phase": "model_catalog",
                "duration_seconds": round(time.monotonic() - started, 3),
                "output_dir": None, "error": error}
            show_state(catalog_errors[name])
        except Exception as exc:
            catalog_errors[name] = exception_state(exc, "model_catalog", started)
            show_state(catalog_errors[name])
else:
    print("NOT_RUN: model catalog lookup is disabled, or configuration needs correction.")


### Optional: select a model for each role

Fill `SELECTED_MODELS` with IDs from `model_options`, or manual IDs when no catalog is available. For example, `{"heavy": "MODEL_ID", "expert": "MODEL_ID"}`. Selection is local and sends no requests. Separate profiles are created for changed roles, so choosing a different lightweight model does not overwrite the retrieval model.


In [ ]:
SELECTED_MODELS = {}  # Example: {"heavy": "MODEL_ID", "expert": "MODEL_ID"}
model_selection_error = None
if SELECTED_MODELS and config is not None:
    profiles = dict(config.profiles)
    roles = dict(config.roles)
    selection_error = None
    for role, model_id in SELECTED_MODELS.items():
        if role not in roles or not isinstance(model_id, str) or not model_id.strip():
            selection_error = {"type": "ModelSelectionError", "code": "INVALID_MODEL_SELECTION",
                "message": "Choose a nonempty model ID for an existing heavy, light, or expert role."}
            break
        source_name = config.roles[role]
        if source_name in model_options and model_id not in {row["id"] for row in model_options[source_name]}:
            selection_error = {"type": "ModelSelectionError", "code": "MODEL_NOT_IN_CATALOG",
                "message": "The selected model ID is absent from the fetched catalog. Select a returned ID or configure a separate manual profile."}
            break
        selected_name = source_name + "__" + role + "_selected"
        profiles[selected_name] = replace(config.profiles[source_name], model=model_id.strip())
        roles[role] = selected_name
    if selection_error:
        model_selection_error = selection_error
        show_state({"status": "FAILED", "phase": "model_selection", "duration_seconds": 0,
                    "output_dir": None, "error": selection_error})
    else:
        config = replace(config, profiles=profiles, roles=roles)
        print(json.dumps(config.to_public_dict(), ensure_ascii=False, indent=2))


## 2. Select a research idea or blind benchmark item

EASY: v0.2 A, N001–N030; MEDIUM: v0.5 A, C001–C030; HARD: v0.5 B, C001–C030.

The dataset loader reads the original complete blind prompt and checks the query/prompt hashes. It never reads gold. For a free-form run, use your own research idea directly. Difficulty labels describe intended difficulty, not a calibrated measurement for every item.


In [ ]:
DIFFICULTY, ITEM_ID = "EASY", "N001"
item = None
query = None
topic = None
input_error = None
EVALUATION_TIER = None
EVALUATION_COMMAND = None
try:
    item = dataset_query(DIFFICULTY, ITEM_ID)
    ITEM_ID = item["item_id"]  # Canonical ID, including case-insensitive user input.
    EVALUATION_TIER = "medium" if DIFFICULTY.casefold() == "mdeium" else DIFFICULTY.casefold()
    EVALUATION_COMMAND = (
        "python scripts/evaluate_public_results.py --tier " + EVALUATION_TIER +
        " --results outputs/notebook-results.jsonl --items " + ITEM_ID +
        " --out outputs/scores.json")
    query = item["prompt"]
    topic = item["query"].split(". ", 1)[0].rstrip(".")
    print(item["query"])
    # Optional free-form input:
    # query = "Describe your research idea here."
    # topic = query
    # item = None  # Free-form ideas are not original benchmark trials.
    # EVALUATION_COMMAND = None
except Exception as exc:
    input_error = exception_state(exc, "benchmark_input")
    show_state(input_error)


In [ ]:
plan = None
if config is not None and query is not None:
    try:
        plan = run_retrieval(query, config, output_dir=ROOT / "outputs/notebook",
                             topic=topic, timeout_seconds=1800, dry_run=True)
    except Exception as exc:
        plan = exception_state(exc, "dry_run_configuration")
    show_state(plan)
else:
    print("NOT_RUN: fix the configuration or benchmark input before continuing.")


## 3. Start real retrieval explicitly

Set `RUN_REAL_REQUESTS = True` only when ready to send API requests. If a routed profile's key is missing from the environment, enter it through a password prompt. A blank key is a configuration failure; it does not start retrieval.

The default Top 5 mode uses a shortlist of up to 60 complete abstracts, five fixed full-text attempts, at most six passages per paper of up to 1,800 characters each, and final Expert ranking. Choose `full_report` for the complete report workflow. Each item stops after 30 minutes by default, with no automatic whole-item rerun.

Failures and partial results display the public error category, safe explanation, available provider code and HTTP status, phase, elapsed time, and artifact directory. Missing keys, unavailable model IDs, endpoint/protocol problems, quotas, and timeouts are distinct operational failures. A catalog entry does not guarantee a successful generation request. Raw provider exceptions and keys are never printed by the helpers below.


In [ ]:
RUN_REAL_REQUESTS = False
result = None
if RUN_REAL_REQUESTS and config is not None and query is not None and not globals().get("model_selection_error"):
    import os
    from getpass import getpass
    started = time.monotonic()
    phase = "credentials"
    try:
        routed = set(config.roles.values())
        profiles = dict(config.profiles)
        missing_key = False
        for name in sorted(routed):
            profile = profiles[name]
            if not profile.api_key and not (profile.api_key_env and os.getenv(profile.api_key_env)):
                entered_key = getpass(f"{name} API Key: ").strip()
                if not entered_key:
                    missing_key = True
                    result = {"status": "FAILED", "phase": "credentials",
                        "duration_seconds": round(time.monotonic() - started, 3),
                        "output_dir": None, "top5": [], "usage": None,
                        "error": {"type": "MissingAPIKey", "code": "MISSING_API_KEY",
                            "message": "Enter an API key for every routed profile before starting retrieval."}}
                    break
                profiles[name] = replace(profile, api_key=entered_key)
                entered_key = None
        if not missing_key:
            config = replace(config, profiles=profiles)
            # config = replace(config, mode="full_report")
            phase = "starting_retrieval"
            def progress(state):
                print(state["phase"], round(state["duration_seconds"], 1), "seconds", end="\r")
            result = run_retrieval(query, config, output_dir=ROOT / "outputs/notebook",
                                   topic=topic, timeout_seconds=1800, progress_callback=progress)
    except EOFError:
        result = {"status": "FAILED", "phase": "credentials",
            "duration_seconds": round(time.monotonic() - started, 3),
            "output_dir": None, "top5": [], "usage": None,
            "error": {"type": "EOFError", "code": "CREDENTIAL_INPUT_UNAVAILABLE",
                "message": "The password prompt is unavailable. Set the profile's API-key environment variable and run this cell again."}}
    except Exception as exc:
        result = exception_state(exc, phase, started)
    print()
    show_state(result)
else:
    print("NOT_RUN: real API requests are disabled, or configuration/input/model selection needs correction.")


## 4. Inspect results and failure details

The summary always shows status, phase, elapsed time, and output directory. Failed and partial runs also show any safe `error.type`, `error.code`, `error.message`, and `error.http_status`. Use the recorded output directory to inspect local artifacts. A failure before a run directory exists reports `output_dir: null`; correct the configuration or input before retrying deliberately.

`usage.reported_token_subtotal` sums tokens reported across stages. If requests omit usage, complete consumption is unknown. Character counts do not replace token counts or establish cost. A returned ranking is distinct from a confirmed benchmark target hit.


In [ ]:
if result is not None:
    show_state(result)
    for paper in result.get("top5", []):
        print(paper.get("rank"), paper.get("title"), paper.get("url", ""))
    print("usage:", json.dumps(result.get("usage"), ensure_ascii=False, indent=2))
else:
    print("NOT_RUN: no real retrieval result is available.")

if item is not None and query == item["prompt"] and EVALUATION_COMMAND:
    print("After saving a real benchmark result, evaluate with:")
    print(EVALUATION_COMMAND)
else:
    print("No benchmark evaluation command: the input is free-form or unavailable.")


## 5. Evaluate offline

Read the separate gold records only after retrieval has finished. Python API results do not automatically include a benchmark ID. For an original benchmark input, save the result with the canonical ID returned by the loader:

```python
if result is not None and item is not None and query == item["prompt"]:
    evaluation_row = dict(result, item_id=item["item_id"])
    evaluation_path = ROOT / "outputs/notebook-results.jsonl"
    evaluation_path.write_text(json.dumps(evaluation_row, ensure_ascii=False) + "\n", encoding="utf-8")
    print(EVALUATION_COMMAND)
```

The preceding result cell derives `EVALUATION_COMMAND` from the selected difficulty and canonical item ID. For MEDIUM/C002 it prints `--tier medium --items C002`; the `MDEIUM` spelling alias also resolves to `medium`. Use the printed command from the repository directory after saving the result. Free-form or modified benchmark input does not produce an original-benchmark evaluation command.

The evaluator does not impute hits for missing or unrun items. Strict designated-target matching and optional work-family matching are separate; title-only identities and historical arXiv versions still require review.

Clear all execution outputs before sharing this Notebook. Commercial use requires a separate license from the project author; see the repository LICENSE and the main [usage guide](../README.md).
